# FLUPS 2: photometric correction function (PCF) from dye references

Template on 260605: the PCF from three dyes (PBBO, C6H, C153). Each dye is measured on the
FLUPS setup (the `PCF/<dye>` runs) and in the fluorimeter (the reference, made by
`flups prep-fluo`). The PCF is reference / measurement, on the detected (upconverted)
wavelength `wl_u`; the dyes cover different ranges and are scaled to each other. The
curve is written as a calibration file that `flups_3_processing` reads.

Two steps, kept in one notebook:

1. Average the PCF runs of each dye: despike, reject bad frames, average, save the means.
2. The PCF from the means and the references: put the reference on `wl_u`, divide, choose
   where each dye is valid, scale the dyes to each other, smooth, write.

There are two decisions: the **validity bounds** of each dye and the **smoothing**. The
rejection of bad frames (`Linear`, one per dye) is interactive. Each interactive tool is
preceded by a **decisions** cell and followed by the cell that uses its result, which calls
`print_decisions()`: paste the printout into the decisions cell. With no clicks, "Run all"
gives the result recorded there; after clicking, paste and re-run before saving. The bounds and the smoothing parameters are literals as well.

Needs `%matplotlib widget` (JupyterLab or VS Code, with `ipympl`).

In [ ]:
%matplotlib widget
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import UnivariateSpline
from scipy.signal import savgol_filter

from edyx.calib import write_calibration
from edyx.fana import despike_median, fluo_wl, pcf_scales, upconvert
from edyx.io import read_dat, stack_traces
from edyx.plot import plot_maps
from pyspan import Linear

## Parameters

In [ ]:
data_dir = Path("../../../edyx/examples/260605")  # edyx example data; in the lab: Path(".")
date = data_dir.resolve().name  # or enter a date manually

dyes = ["PBBO", "C6H", "C153"]  # blue to red; the first dye is the reference of the scales
runs = {  # FLUPS runs of each dye, in data_dir / "PCF" / dye
    "PBBO": [f"260605_PCF_PBBO_run{i:02d}.dat" for i in (6, 7, 8, 9)],  # choice: runs at 100 ps
    "C6H": [f"260605_PCF_C6H_run{i:02d}.dat" for i in (1, 2, 3)],
    "C153": [f"260605_PCF_C153_run{i:02d}.dat" for i in (1, 2, 3)],
}
ref_files = {  # fluorimeter reference of each dye, in data_dir / "PCF" (made by flups prep-fluo)
    "PBBO": "260605_PBBO_ac_all_corr.dat",
    "C6H": "260605_C6H_ac_1mm_corr.dat",
    "C153": "260605_C153_ac_1mm_corr.dat",
}
gate = 1340  # nm, gate wavelength of the experiment
despike_size = (15, 5)  # choice: median filter size (wavelengths, frames)
despike_threshold = 100  # choice: see "Read and despike"

out_dir = Path(".")
pcf_file = out_dir / f"{date}_pcf_dyes.toml"

## Step 1: average the PCF runs of each dye

Despike, reject bad frames, average; the means are saved as `<date>_PCF_<dye>_mean.dat`
(columns: `wl_u`, counts).

### Read and despike

Each run is one delay with several frames, axes `(wl_u, frame)`. The runs of a dye are
stacked with `stack_traces`.

`despike_median` replaces the points that differ from the local median by more than the
threshold. Recipe for the threshold: about 2 x (p95 - p5) of a region without fluorescence, here
`wl_u` 480-520 nm (`wl_f` above 740 nm), pooled over all the runs used. The commented lines
in the next cell give 140 on these data; the literal above is the choice. The map below shows the replaced
points (raw - despiked; the horizontal axis is the frame number, labelled "delay" by `plot_maps`).

In [ ]:
stacks = {dye: stack_traces([read_dat(data_dir / "PCF" / dye / name) for name in runs[dye]])
          for dye in dyes}
wl_u = stacks[dyes[0]][1]
raw = {dye: np.ma.getdata(s) for dye, (_, _, s) in stacks.items()}  # (run, wl_u, frame)

# bg = np.concatenate([raw[dye][:, (wl_u > 480) & (wl_u < 520)].ravel() for dye in dyes])
# 2 * np.subtract(*np.percentile(bg, [95, 5]))  # -> 140 on these data
despiked = {dye: np.array([despike_median(s, despike_size, threshold=despike_threshold)
                           for s in raw[dye]]) for dye in dyes}
{dye: int((despiked[dye] != raw[dye]).sum()) for dye in dyes}  # replaced points

In [ ]:
removed = np.concatenate([raw[dye] - despiked[dye] for dye in dyes])  # (all runs, wl_u, frame)
titles = [name[11:-4] for dye in dyes for name in runs[dye]]
lim = 2 * despike_threshold
fig, axes = plot_maps(np.arange(removed.shape[-1]), wl_u, removed, titles=titles, ncols=4,
                      vmin=-lim, vmax=lim)

### Reject bad frames and average the runs

`Linear` per dye: the runs are the files, the frames the repetitions. RMSD of each frame
against the median of its run; frames above the threshold are rejected. The mean of the kept
frames of each run is averaged over the runs: `meas[dye]` on `wl_u`.

In [ ]:
# Paste the threshold printed for each dye by print_decisions() here. [] = default threshold
thresholds = {"PBBO": [991.27, 1658.97, 1056.39, 935.99], "C6H": [1511.12, 2321.88, 1597.96], "C153": [1156.83, 469.87, 570.33]}

In [ ]:
lins = {}
lins["PBBO"] = Linear(wl=wl_u, od=despiked["PBBO"], fnames=runs["PBBO"], threshold=thresholds["PBBO"])
lins["PBBO"].plot()

In [ ]:
lins["C6H"] = Linear(wl=wl_u, od=despiked["C6H"], fnames=runs["C6H"], threshold=thresholds["C6H"])
lins["C6H"].plot()

In [ ]:
lins["C153"] = Linear(wl=wl_u, od=despiked["C153"], fnames=runs["C153"], threshold=thresholds["C153"])
lins["C153"].plot()

In [ ]:
for dye, l in lins.items(): 
    print(f"{dye:5} ", end="")
    l.print_decisions()

meas = {dye: lins[dye].mean_spectra.mean(axis=0) for dye in dyes}  # counts, on wl_u

for dye in dyes:  # the means, saved: the result of step 1
    np.savetxt(out_dir / f"{date}_PCF_{dye}_mean.dat", np.column_stack([wl_u, meas[dye]]),
               fmt="%.6e", header=f"wl_u / nm, counts; mean of {runs[dye]}")

## Step 2: PCF from the means and the references

Starts from `meas` and `wl_u` of step 1. To start here from the saved means instead:

In [ ]:
# means = {dye: np.loadtxt(out_dir / f"{date}_PCF_{dye}_mean.dat") for dye in dyes}
# wl_u = means[dyes[0]][:, 0]
# meas = {dye: m[:, 1] for dye, m in means.items()}

### Reference spectra

The fluorimeter references from `flups prep-fluo`: two columns, `wl_f` (nm) and intensity.
Instead of a measured reference, the lognormal fits of Gerecke et al. are available for C6H,
C153 and DCM (none for PBBO): `ref_f["C153"] = np.column_stack([wl_g, ref_spec_wl(wl_g, "C153")])`
with `from edyx.fana import ref_spec_wl` and `wl_g` a fluorescence wavelength grid, e.g.
`np.arange(300, 700.1, 1)`.

We then convert the reference and interpolate it on the FLUPS `wl_u` axis. Both
are normalized.

In [ ]:
ref_f = {dye: np.loadtxt(data_dir / "PCF" / ref_files[dye]) for dye in dyes}  # columns: wl_f, intensity
# ref_f["C153"] = np.column_stack([wl_g, ref_spec_wl(wl_g, "C153")])  # alternative, see above

ref_u, meas_n = {}, {}
for dye in dyes:
    wl_r, r = upconvert(ref_f[dye][:, 0], ref_f[dye][:, 1]) # upconvert reference
    s = np.interp(wl_u, wl_r, r, left=np.nan, right=np.nan) # interpolate onto wl_u
    ref_u[dye] = s / np.nanmax(s)
    meas_n[dye] = meas[dye] / np.max(savgol_filter(meas[dye], 11, 2)) # normalize to max of smoothed version.

### Decisions: validity bounds and smoothing parameters.

The PCF of a dye is `ref_u / meas_n`. It is only meaningful where both spectra are well above
the noise, so each dye gets **validity bounds** in nm of `wl_u`. The red edge of a fluorescence spectrum is
the more reliable one (reabsorption, inner filter on the blue side). Look at the spectra (top,
shaded: the bounds) and the curves (bottom, bold inside the bounds) and adjust the bounds
until the dyes agree where they overlap.

Chosen here, from these data:

- PBBO from 285 nm, where its reference starts, to 355 nm (above, the measured band is at the
  noise level and the ratio blows up);
- C6H 345 to 400 nm (below, the measured rise of the band lags the reference; above, noise);
- C153 375 to 425 nm (below, the same lag on the blue edge; the band at 445 nm in the measured
  spectra is an artifact, seen in C6H as well).

The overlaps are PBBO-C6H at 345-355 nm and C6H-C153 at 375-400 nm.

Then, we scale pcfs to one another. `pcf_scales` minimizes the squared differences
between the scaled curves where two dyes overlap, with the first dye fixed to 1 
(closed form).

Finally, a smooth curve goes through the data. This occurs in two steps:
first we use a smoothing spline through all the curves, then we smooth *that*
using Savitzky-Golay. The **smoothing parameters** are therefore a smoothing
factor for the spline, then the window and degree parameters of SG.

In [ ]:
bounds = {"PBBO": (285, 355), "C6H": (345, 400), "C153": (375, 425)}  # choice: nm of wl_u

factor, window, degree = 0.003, 21, 3  # choice: spline smoothing per point, filter window and order

inside = {dye: (wl_u >= bounds[dye][0]) & (wl_u <= bounds[dye][1]) for dye in dyes}
curve = {dye: ref_u[dye] / meas_n[dye] for dye in dyes}
curves = {dye: np.where(inside[dye], curve[dye], np.nan) for dye in dyes}

scales = dict(zip(dyes, pcf_scales(np.vstack([curves[dye] for dye in dyes]))))
scaled = {dye: curves[dye] * scales[dye] for dye in dyes}
print({dye: round(float(s), 3) for dye, s in scales.items()})

# collect and smooth pcf
x = np.concatenate([wl_u[np.isfinite(scaled[dye])] for dye in dyes])
y = np.concatenate([scaled[dye][np.isfinite(scaled[dye])] for dye in dyes])
srt = np.argsort(x)
x, y = x[srt], y[srt]
spline = UnivariateSpline(x, y, s=factor * len(x))
valid = (wl_u >= x[0]) & (wl_u <= x[-1])
pcf = np.full_like(wl_u, np.nan)
pcf[valid] = savgol_filter(spline(wl_u[valid]), window, degree)
print(f"PCF on {wl_u[valid][0]:.1f}-{wl_u[valid][-1]:.1f} nm, min {np.nanmin(pcf):.3f}, max {np.nanmax(pcf):.3f}")


In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 8), layout="constrained")
for i, dye in enumerate(dyes):
    axes[0].plot(wl_u, ref_u[dye], color=f"C{i}", label=f"{dye} reference")
    axes[0].plot(wl_u, meas_n[dye], color=f"C{i}", lw=0.5, alpha=0.6, label=f"{dye} measured")
    axes[0].axvspan(*bounds[dye], color=f"C{i}", alpha=0.1)
    axes[1].plot(wl_u, curve[dye], color=f"C{i}", lw=0.5, alpha=0.3)
    axes[1].plot(wl_u, curves[dye], color=f"C{i}", label=dye)
axes[0].set_ylim(-0.2, 1.2)
axes[1].set_yscale("log")
axes[1].set_ylim(0.2, 5)
axes[1].set_xlim(260, 460)
axes[1].set_xlabel(r"$\lambda_\mathrm{u}$ / nm")
axes[0].set_ylabel("spectra, normalized")
axes[1].set_ylabel("reference / measured")
axes[0].legend(fontsize="small", ncols=3)
axes[1].legend()

for i, dye in enumerate(dyes):
    axes[2].plot(wl_u, scaled[dye], color=f"C{i}", lw=0.5, alpha=0.5, label=dye)
axes[2].plot(wl_u[valid], spline(wl_u[valid]), color="C3", lw=0.5, label="spline")
axes[2].plot(wl_u, pcf, color="k", label="PCF")
axes[2].set_yscale("log")
axes[2].set_xlim(260, 460)
axes[2].set_ylim(0.3, 5)
axes[2].set_xlabel(r"$\lambda_\mathrm{u}$ / nm")
axes[2].set_ylabel("PCF")
axes[2].legend()

### Write the PCF

The calibration file holds the curve on `wl_u`, NaN outside the valid range
(`pcf_correct` in `flups_3_processing` masks those points). The dyes, bounds, scales and
smoothing are kept for information.

In [ ]:
out_dir.mkdir(exist_ok=True)
write_calibration(pcf_file, "pcf", "wl_u", wl_u, pcf, units=["nm", ""], date=date,
                  source=", ".join(dyes), made_by="flups_2_pcf.ipynb",
                  fit={"dyes": dyes,
                       "bounds": {dye: list(bounds[dye]) for dye in dyes},
                       "scales": scales,
                       "reference": ref_files,
                       "gate": gate,
                       "smoothing": {"factor": factor, "window": window, "degree": degree}})
pcf_file.name